# 최종 결과 검증 (double check)

`5.0-aggregate.ipynb`가 만든 `results/.../school_counts.csv`를 제출하기 전에,
사람이 1000줄을 다 눈으로 보지 않고도 자동으로 확인할 수 있는 두 가지 체크를 한다.

1. **총량 검증**: raw 댓글 수(1000) 대비 최종 집계 합계가 어떻게 구성되는지 breakdown
   (0건/1건/2건 이상 매칭 댓글이 각각 몇 개고, 합계가 1000과 왜 다른지 설명 가능해야 함).
2. **구조 검증**: 최종 학교명에 지역/고유명사 접두 어절이 없는 경우(예: "초등학교" 단독)를
   자동으로 플래그. `data/processed/gt_schoolnames.csv` 전체(12,439건) 기준으로,
   접미사(초등학교/중학교/고등학교/대학교 등) 앞 접두 어절이 2글자 미만인 행은
   버그로 보이는 딱 1건("초등학교" 그 자체) 뿐이라는 걸 확인했음 -> "접두 어절 >= 2글자"를
   실제 학교명의 필요조건으로 삼아도 과검출이 없음.

In [ ]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("repo 루트를 못 찾았어요 (data/, src/ 폴더 기준)")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src" / "utils") not in sys.path:
    sys.path.append(str(REPO_ROOT / "src" / "utils"))

REPO_ROOT

## 1. 총량 검증 — raw 댓글 수 vs 최종 집계 합계

In [ ]:
import pandas as pd

raw_df = pd.read_csv(REPO_ROOT / "data" / "raw" / "dataset.csv", encoding="utf-8-sig")
match_df = pd.read_csv(
    REPO_ROOT / "data" / "interim" / "gt_match_results.csv", encoding="utf-8-sig"
)
match_df["ans"] = match_df["ans"].fillna("")
match_df["ans_count"] = match_df["ans_count"].fillna(0).astype(int)

school_counts = pd.read_csv(
    REPO_ROOT / "results" / "school_counts_2026-08-22" / "school_counts.csv",
    encoding="utf-8-sig",
    index_col=0,
)["count"]

n_raw = len(raw_df)
n_matched_rows = len(match_df)
assert n_raw == n_matched_rows, (
    f"raw 댓글 수({n_raw})와 gt_match_results 행 수({n_matched_rows})가 다름 "
    "-- 파이프라인 어딘가에서 행이 사라지거나 늘어남. 4.0/5.0 노트북 재확인 필요."
)

n_zero = int((match_df["ans_count"] == 0).sum())
n_one = int((match_df["ans_count"] == 1).sum())
n_two_plus = int((match_df["ans_count"] >= 2).sum())
contribution_zero = 0
contribution_one = n_one
contribution_two_plus = int(match_df.loc[match_df["ans_count"] >= 2, "ans_count"].sum())
total_contribution = contribution_zero + contribution_one + contribution_two_plus

print(f"raw 댓글 수: {n_raw}")
print(f"  - 학교 0개 매칭: {n_zero}건 -> 최종 합계 기여 0")
print(f"  - 학교 1개 매칭: {n_one}건 -> 최종 합계 기여 {contribution_one}")
print(
    f"  - 학교 2개 이상 매칭: {n_two_plus}건 -> 최종 합계 기여 {contribution_two_plus} "
    "(다중 학교 언급을 전부 카운트하는 현재 정책 때문에 건수보다 더 크게 기여함)"
)
print(f"기대 총합(0+1+2건 기여 합): {total_contribution}")

final_sum = int(school_counts.sum())
print(f"실제 school_counts.csv 합계: {final_sum}")

assert final_sum == total_contribution, (
    f"school_counts.csv 합계({final_sum})가 gt_match_results 기준 기대값({total_contribution})과 "
    "다름 -- 집계 단계(5.0)에서 무언가 누락/중복됐을 가능성."
)

if final_sum != n_raw:
    print(
        f"\n[알림] 최종 합계({final_sum})가 raw 댓글 수({n_raw})와 다름. "
        f"차이 {final_sum - n_raw}건은 '다중 학교 언급 댓글을 전부 카운트'하는 현재 정책 때문 "
        "(요구사항: 댓글당 1회만 인정할지 여부를 팀/제출 기준으로 정해야 함 -- 이 노트북은 "
        "그 규모를 자동으로 드러내는 역할만 하고, 정책 결정은 하지 않음)."
    )

## 2. 구조 검증 — 학교명에 접두 어절(고유명사)이 있는지 확인

실제 학교명은 항상 `[지역/고유명사 2글자 이상] + [학교급 접미사]` 형태다.
GT 사전 12,439건 전체에서 이 규칙을 어긴 행은 버그로 보이는 "초등학교" 단독 1건뿐이었음
(직접 계산해서 확인한 임계값 -- 감으로 정한 게 아님). 이 규칙을 최종 집계 결과에도 적용해서
"학교 유형 단어만 있고 앞에 아무것도 없는" 오검출을 자동으로 잡는다.

In [ ]:
# 긴 접미사부터 검사해야 "대학교"가 "학교"로 먼저 걸려서 접두 길이가 잘못 계산되는 일이 없음
SCHOOL_TYPE_SUFFIXES = ["초등학교", "중학교", "고등학교", "대학교", "대학", "학교"]


def prefix_len(name: str):
    for suf in SCHOOL_TYPE_SUFFIXES:
        if name.endswith(suf):
            return len(name) - len(suf)
    return None  # 학교 유형 접미사 자체가 없으면 애초에 학교명이 아닌 것으로 간주


def is_structurally_valid(name: str) -> bool:
    plen = prefix_len(name)
    return plen is not None and plen >= 2


invalid_names = [name for name in school_counts.index if not is_structurally_valid(name)]

if invalid_names:
    print("[구조 검증 실패] 접두 어절이 없거나 너무 짧은 학교명 발견:")
    for name in invalid_names:
        print(f"  - '{name}' (count={school_counts[name]})")
        # 어떤 원본 댓글들 때문에 이 이름이 나왔는지 역추적
        source_rows = match_df[
            match_df["ans"].apply(lambda ans: name in ans.split())
        ][["comment_id", "comment", "school_candidate"]]
        print(source_rows.to_string(index=False))
        print()
else:
    print("[구조 검증 통과] 모든 최종 학교명이 접두 어절 >= 2글자 규칙을 만족함.")

## 3. 최하위 count 스팟체크 리스트

count가 낮은 항목일수록 오탐/노이즈일 확률이 높음 (전체 1000줄을 다 보는 대신,
위험도가 높은 하위권만 우선순위로 사람이 확인).

In [ ]:
SPOT_CHECK_N = 10

lowest = school_counts.sort_values(ascending=True).head(SPOT_CHECK_N)
print(f"count 하위 {SPOT_CHECK_N}개 (사람이 직접 스팟체크 권장):")
print(lowest.to_string())

## 검증 리포트 저장

In [ ]:
report_path = REPO_ROOT / "results" / "school_counts_2026-08-01" / "validation_report.txt"

lines = []
lines.append("=== 총량 검증 ===")
lines.append(f"raw 댓글 수: {n_raw}")
lines.append(f"학교 0개 매칭: {n_zero}건")
lines.append(f"학교 1개 매칭: {n_one}건")
lines.append(f"학교 2개 이상 매칭: {n_two_plus}건 (합계 기여 {contribution_two_plus})")
lines.append(f"최종 school_counts.csv 합계: {final_sum} (raw 대비 차이: {final_sum - n_raw})")
lines.append("")
lines.append("=== 구조 검증 ===")
if invalid_names:
    for name in invalid_names:
        lines.append(f"[FLAG] '{name}' count={school_counts[name]} -- 접두 어절 없음/부족")
else:
    lines.append("이상 없음")
lines.append("")
lines.append(f"=== count 하위 {SPOT_CHECK_N}개 스팟체크 대상 ===")
for name, count in lowest.items():
    lines.append(f"{name}\t{count}")

report_path.write_text("\n".join(lines), encoding="utf-8")
report_path